In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision.transforms import transforms
import torchtext


In [ ]:
torch.manual_seed(1234)
torch.cuda.manual_seed(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
train_data = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_size = int(0.8 * len(train_data))
valid_size = len(train_data) - train_size
train_data, valid_data = random_split(train_data, [train_size, valid_size])

In [ ]:
test_data.data.shape, test_data.targets.shape

(torch.Size([10000, 28, 28]), torch.Size([10000]))

In [ ]:
# test_data[0]

In [ ]:
train_data.dataset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5,), std=(0.5,))
           )

In [ ]:
# train_data[0]

In [ ]:
BATCH_SIZE = 64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class SimpleNN(nn.Module):
  def __init__(self):
    super(SimpleNN, self).__init__()
    self.fc1 = nn.Linear(28*28, 512)
    self.fc2 = nn.Linear(512, 256)
    self.fc3 = nn.Linear(256, 10)
    self.dropout = nn.Dropout(0.5)

  def forward(self, x):
    x = x.view(-1, 28*28) # flatten
    x = torch.relu(self.fc1(x))
    x = self.dropout(x)
    x = torch.relu(self.fc2(x))
    x = self.dropout(x)
    x = torch.relu(self.fc3(x))
    return x



In [ ]:
model = SimpleNN()
model

SimpleNN(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [ ]:
for p in model.parameters():
  print(type(p), p.size())
  print(p.numel())

<class 'torch.nn.parameter.Parameter'> torch.Size([512, 784])
401408
<class 'torch.nn.parameter.Parameter'> torch.Size([512])
512
<class 'torch.nn.parameter.Parameter'> torch.Size([256, 512])
131072
<class 'torch.nn.parameter.Parameter'> torch.Size([256])
256
<class 'torch.nn.parameter.Parameter'> torch.Size([10, 256])
2560
<class 'torch.nn.parameter.Parameter'> torch.Size([10])
10


In [ ]:
# help(model)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

criterion.to(device)
model.to(device)

SimpleNN(
  (fc1): Linear(in_features=784, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [ ]:
N_EPOCH = 2

for epoch in range(N_EPOCH):
  model.train()
  train_loss = 0
  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    out = model(images)
    loss = criterion(out, labels)
    loss.backward()
    optimizer.step()
    train_loss += loss.item()

  val_loss = 0
  model.eval()
  with torch.no_grad():
    for images, labels in valid_loader:
      images, labels = images.to(device), labels.to(device)
      out = model(images)
      loss = criterion(out, labels)
      val_loss += loss.item()

  print(f"Epoch: {epoch}, Train Loss: {train_loss/len(train_loader)}, Valid Loss: {val_loss/len(valid_loader)}")

Epoch: 0, Train Loss: 0.5746847819288572, Valid Loss: 0.23707408303434543
Epoch: 1, Train Loss: 0.3265129840373993, Valid Loss: 0.18414201084128085
Epoch: 2, Train Loss: 0.29150869518021744, Valid Loss: 0.1712167570664686
Epoch: 3, Train Loss: 0.25909077529112495, Valid Loss: 0.15400315185097305
Epoch: 4, Train Loss: 0.23995054307579994, Valid Loss: 0.14517358412094256
Epoch: 5, Train Loss: 0.2296934624115626, Valid Loss: 0.13342755791196165
Epoch: 6, Train Loss: 0.22025243031481903, Valid Loss: 0.13040589242975445
Epoch: 7, Train Loss: 0.21282589488476514, Valid Loss: 0.11642794841107854
Epoch: 8, Train Loss: 0.19358388273169597, Valid Loss: 0.11311291242414649
Epoch: 9, Train Loss: 0.20409333177407582, Valid Loss: 0.11545224357990826


In [ ]:
model.eval()
test_loss = 0
tot = 0
corr = 0
with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    test_loss += loss
    _, pred = torch.max(outputs, 1)
    # print(pred)
    tot += len(labels)
    corr += (pred == labels).sum().item()
print(f"Test Loss: {test_loss/len(test_loader)}, Test Accuracy: {100 * corr / tot}")


Test Loss: 0.11197219789028168, Test Accuracy: 96.53
